In [1]:
%pip install --upgrade pip
!pip3 install pyreclab --upgrade

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pyreclab (from versions: none)
ERROR: No matching distribution found for pyreclab


In [2]:
%pip install matplotlib
%pip install tensorflow
%pip install imblearn
%pip install scipy
%pip install numpy
%pip install scikit-learn
%pip install seaborn 
%pip install pandas
%pip install setuptools
%pip install distutils
%pip install importlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement distutils (from versions: none)
ERROR: No matching distribution found for distutils



  Using cached importlib-1.0.4.zip (7.1 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [1 lines of output]
      ERROR: Can not execute `setup.py` since setuptools is not available in the build environment.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [3]:
import os
import pandas as pd
# import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
# import scipy.sparse as sparse
# from imblearn.datasets import make_imbalance
# from collections import Counter
# from tensorflow.keras.layers import BatchNormalization

import tensorflow as tf


%matplotlib inline
sns.set(style="whitegrid")

In [ ]:
!curl -L -o archive.zip https://www.kaggle.com/api/v1/datasets/download/retailrocket/ecommerce-dataset

In [ ]:
!unzip archive.zip -d ecommerce-dataset

In [4]:
import zipfile

with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('ecommerce-dataset')

In [5]:
# Ruta de la carpeta descomprimida
output_folder = 'ecommerce-dataset'

# Cargar los archivos CSV en DataFrames
category_tree = pd.read_csv(os.path.join(output_folder, 'category_tree.csv'))
events = pd.read_csv(os.path.join(output_folder, 'events.csv'))
item_properties_part1 = pd.read_csv(os.path.join(output_folder, 'item_properties_part1.csv'))
item_properties_part2 = pd.read_csv(os.path.join(output_folder, 'item_properties_part2.csv'))

In [6]:
# Visualizamos los eventos
print(events.head())

       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN


Preprocesamiento

In [7]:
# Testeamos el modelo con un dataset de menor fraccion
dataset = events.sample(frac=0.01, random_state=42)
dataset = dataset.drop('transactionid', axis=1)
dataset = dataset.drop('timestamp', axis=1)

print(dataset.head())
print(dataset.shape)

         visitorid event  itemid
486798       50734  view    4442
1145255     355903  view  269631
1601366    1066758  view  221329
843976     1049477  view   23683
2524686     143239  view    6552
(27561, 3)


In [8]:
test = events.drop(dataset.index)
# Seleccionar los datos de prueba excluyendo el entrenamiento
test = test.sample(frac=0.01, random_state=42)
test = test.drop('transactionid', axis=1)
test = test.drop('timestamp', axis=1)
print(test.shape)

(27285, 3)


In [9]:
# Verificamos que la cantidad de filas y columnas no exceda el millon de entradas
num_rows = len(dataset['visitorid'].unique())
num_columns = len(dataset['itemid'].unique())

print(f"Número de usuarios únicos: {num_rows}")
print(f"Número de ítems únicos: {num_columns}")

Número de usuarios únicos: 25669
Número de ítems únicos: 19971


In [10]:
duplicados = dataset.duplicated(subset=['visitorid', 'itemid'])
print(f"Cantidad de filas duplicadas: {duplicados.sum()}")

duplicados = test.duplicated(subset=['visitorid', 'itemid'])
print(f"Cantidad de filas duplicadas TEST: {duplicados.sum()}")

Cantidad de filas duplicadas: 189
Cantidad de filas duplicadas TEST: 156


In [11]:
dataset = dataset.drop_duplicates(subset=['visitorid', 'itemid'])
dataset = dataset.dropna(subset=['visitorid', 'itemid'])

test = test.drop_duplicates(subset=['visitorid', 'itemid'])
test = test.dropna(subset=['visitorid', 'itemid'])

In [12]:
duplicados = dataset.duplicated(subset=['visitorid', 'itemid'])
print(f"Cantidad de filas duplicadas: {duplicados.sum()}")

duplicados = test.duplicated(subset=['visitorid', 'itemid'])
print(f"Cantidad de filas duplicadas TEST: {duplicados.sum()}")

Cantidad de filas duplicadas: 0
Cantidad de filas duplicadas TEST: 0


In [13]:
# Agregmos los pesos a los eventos respectivamente
pesos = {
    "view": 1,
    "addtocart": 2,
    "transaction": 3
}

dataset['peso'] = dataset['event'].apply(lambda x: pesos.get(x, 0))
dataset = dataset.dropna(subset=['visitorid', 'itemid', 'peso'])
print(dataset.head())

         visitorid event  itemid  peso
486798       50734  view    4442     1
1145255     355903  view  269631     1
1601366    1066758  view  221329     1
843976     1049477  view   23683     1
2524686     143239  view    6552     1


In [14]:
# Agregmos los pesos a los eventos respectivamente (TEST)
pesos = {
    "view": 1,
    "addtocart": 2,
    "transaction": 3
}

test['peso'] = test['event'].apply(lambda x: pesos.get(x, 0))
test = test.dropna(subset=['visitorid', 'itemid', 'peso'])
print(dataset.head())

         visitorid event  itemid  peso
486798       50734  view    4442     1
1145255     355903  view  269631     1
1601366    1066758  view  221329     1
843976     1049477  view   23683     1
2524686     143239  view    6552     1


In [15]:

dataset = dataset.replace(-1, 0)
print(dataset.shape)

test = test.replace(-1, 0)
print(test.shape)

(27372, 4)
(27129, 4)


In [16]:
expected_shape = (dataset['visitorid'].nunique(), dataset['itemid'].nunique())
print(f"Tamaño esperado: {expected_shape}")
print(f"Total de elementos: {expected_shape[0] * expected_shape[1]}")


Tamaño esperado: (25669, 19971)
Total de elementos: 512635599


In [17]:
test = test.sample(frac=0.2, random_state=42)

In [18]:
trn_pv = dataset.pivot_table(index='visitorid', columns='itemid', values='peso', fill_value=0)


In [19]:
tst_pv = test.pivot_table(index='visitorid', columns='itemid', values='peso', fill_value=0)

In [20]:
all_pv = pd.concat([dataset, test]).drop_duplicates(subset=['visitorid', 'itemid'])
all_pv = all_pv.pivot_table(index='visitorid', columns='itemid', values='peso')


In [22]:
from scipy.sparse import csr_matrix

# Crear matriz dispersa
data = pd.concat([dataset, test])
row = data['visitorid'].astype('category').cat.codes
col = data['itemid'].astype('category').cat.codes
sparse_matrix = csr_matrix((data['peso'], (row, col)))

# Explorar tamaño y eficiencia
print(sparse_matrix.shape)


(30547, 22896)


In [23]:
# Crear matriz dispersa
row = dataset['visitorid'].astype('category').cat.codes
col = dataset['itemid'].astype('category').cat.codes
sparse_trn = csr_matrix((dataset['peso'], (row, col)))

# Explorar tamaño y eficiencia
print(sparse_matrix.shape)

(30547, 22896)


In [24]:
# Crear matriz dispersa
row = test['visitorid'].astype('category').cat.codes
col = test['itemid'].astype('category').cat.codes
sparse_tst = csr_matrix((test['peso'], (row, col)))

# Explorar tamaño y eficiencia
print(sparse_matrix.shape)

(30547, 22896)


In [26]:
R = sparse_matrix
sparse_mask_R = (sparse_matrix != 0).astype(int)
sparse_train_mask_R = (sparse_trn != 0).astype(int)
sparse_test_mask_R = (sparse_tst != 0).astype(int)


In [29]:
user_train_set = set(dataset['visitorid'])
item_train_set = set(dataset['itemid'])

user_test_set = set(test['visitorid'])
item_test_set = set(test['itemid'])

num_users, num_items = R.shape
num_train_ratings = len(dataset)
num_test_ratings = len(test)
num_total_ratings = num_train_ratings + num_test_ratings


Preparar Modelo

In [30]:
## parameter setting
hidden_neuron = 50
# layer_structure = [num_items, 512, 128, hidden_neuron, 128, 512, num_items]
layer_structure = [num_items, 128, hidden_neuron, 128, num_items]
lambda_value = 1e-3
lr = 1e-3
global_step = tf.Variable(0, trainable=False)
min_RMSE = 99999
min_epoch = -99999
patience = 0
total_patience = 20

In [32]:
batch_size = 64
batch_data = tf.random.uniform((batch_size, num_items), dtype=tf.float32)

# Usar tensores directamente
model_mask_corruption = batch_data  # Por ejemplo, una máscara
input_R = batch_data  # Datos reales
input_mask_R = tf.cast(batch_data > 0, tf.float32)  # Máscara binaria

real_batch_size = tf.shape(input_R)[0]
model_batch_data_idx = tf.range(start=0, limit=real_batch_size, dtype=tf.int32)



In [33]:
corrupted_R = tf.multiply(model_mask_corruption, input_R)
corrupted_input_mask_R = tf.multiply(model_mask_corruption, input_mask_R)

In [35]:
# Crear una variable para las embeddings de usuario
V = tf.Variable(
    initial_value=tf.keras.initializers.GlorotUniform()(shape=[num_users, layer_structure[1]]),
    trainable=True,
    name="User_embed",
    dtype=tf.float32
)

# Usar embedding_lookup para seleccionar embeddings específicos
batch_V = tf.nn.embedding_lookup(V, model_batch_data_idx)

In [37]:
# Función para inicializar los pesos y sesgos de cada capa
def make_layer_weights(n_visible, n_hidden, itr):
    # Crear pesos con inicialización Glorot (equivalente a Xavier)
    pre_W = tf.Variable(
        initial_value=tf.keras.initializers.GlorotUniform()(shape=[n_visible, n_hidden]),
        trainable=True,
        name=f"pre_W{itr}",
        dtype=tf.float32
    )
    # Crear sesgos inicializados en ceros
    pre_b = tf.Variable(
        initial_value=tf.zeros(shape=[n_hidden], dtype=tf.float32),
        trainable=True,
        name=f"pre_b{itr}",
        dtype=tf.float32
    )
    return pre_W, pre_b

# Estructura de la red
n_layer = len(layer_structure)
Weight = dict()
bias = dict()

# Crear pesos y sesgos para cada capa
for itr in range(n_layer - 1):
    Weight[itr], bias[itr] = make_layer_weights(layer_structure[itr], layer_structure[itr + 1], itr)

# Imprimir un ejemplo de los pesos y sesgos creados (opcional)
print("Peso de la capa 0:", Weight[0])
print("Sesgo de la capa 0:", bias[0])

Peso de la capa 0: <tf.Variable 'pre_W0:0' shape=(22896, 128) dtype=float32, numpy=
array([[-0.00191858, -0.01523701,  0.00567738, ...,  0.01492775,
        -0.0130719 ,  0.0072198 ],
       [ 0.00362881,  0.00791646,  0.00979035, ...,  0.00415976,
         0.00897762, -0.00191532],
       [-0.01289634,  0.00389292,  0.0012101 , ..., -0.00564105,
         0.01572712,  0.00357216],
       ...,
       [-0.01215114, -0.01457334, -0.00300821, ...,  0.01293985,
         0.00577097, -0.00882609],
       [-0.00942843, -0.00545463, -0.00357913, ...,  0.00189714,
         0.00045703,  0.00571086],
       [ 0.00323862, -0.0039545 , -0.00928238, ..., -0.01011152,
         0.00657975, -0.00349125]], dtype=float32)>
Sesgo de la capa 0: <tf.Variable 'pre_b0:0' shape=(128,) dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

Encoder + Decoder

In [38]:
batch_normalization = 'FALSE'
f_act = tf.nn.sigmoid
g_act = tf.nn.sigmoid
keep_prob = 1.0

hidden_value = corrupted_R

In [ ]:
# Configuración de batch normalization y funciones de activación
batch_normalization = True  # Cambiar según necesidad
f_act = tf.nn.sigmoid  # Activación para la parte del encoder
g_act = tf.nn.sigmoid  # Activación para la parte del decoder
keep_prob = 0.5  # Probabilidad de mantener nodos durante dropout

# Inicialización de valores previos
hidden_value = batch_V  # Valor inicial del input
Encoded_X = None  # Para guardar la salida del encoder

# Iterar sobre las capas
for itr1 in range(len(layer_structure) - 1):
    # **Encoder**
    if itr1 <= int(len(layer_structure) / 2) - 1:
        if itr1 == 0:
            before_activation = tf.add(
                tf.matmul(hidden_value, Weight[itr1]) + batch_V, bias[itr1]
            )
        else:
            before_activation = tf.add(tf.matmul(hidden_value, Weight[itr1]), bias[itr1])
        
        if batch_normalization:
            before_activation = tf.keras.layers.BatchNormalization()(before_activation)
        
        hidden_value = f_act(before_activation)
    
    # **Decoder**
    elif itr1 > int(len(layer_structure) / 2) - 1:
        before_activation = tf.add(tf.matmul(hidden_value, Weight[itr1]), bias[itr1])
        
        if batch_normalization:
            before_activation = tf.keras.layers.BatchNormalization()(before_activation)
        
        hidden_value = g_act(before_activation)
    
    # **Dropout en todas las capas excepto la última**
    if itr1 < len(layer_structure) - 2:
        hidden_value = tf.keras.layers.Dropout(rate=1 - keep_prob)(hidden_value)
    
    # **Guardar salida del Encoder**
    if itr1 == int(len(layer_structure) / 2) - 1:
        Encoded_X = hidden_value

# Salida final del Decoder
Decoder = hidden_value

# Mostrar resultados (opcional)
print("Salida del Encoder (Encoded_X):", Encoded_X)
print("Salida del Decoder:", Decoder)


Optimizacion

In [ ]:
## avg reconstruction error term
### log cross entropy
pre_cost1 = -1 * tf.multiply(corrupted_R, tf.log(Decoder)) - \
                tf.multiply((1-corrupted_R) , tf.log(1-Decoder))
### corrupted_input_mask_R 
pre_cost1 = tf.multiply(pre_cost1, corrupted_input_mask_R)
### error average
cost1 = tf.reduce_sum(pre_cost1) / tf.cast(real_batch_size, tf.float32)

In [ ]:
## regularization term
### weight paramter l2 norm
pre_cost2 = tf.constant(0, dtype=tf.float32)
for itr in range(len(Weight.keys())):
    pre_cost2 = tf.add(pre_cost2,
                       tf.add(tf.nn.l2_loss(Weight[itr]), tf.nn.l2_loss(bias[itr])))
pre_cost2 = pre_cost2 + tf.nn.l2_loss(batch_V)
### lambda value
cost2 = lambda_value * 0.5 * pre_cost2

In [ ]:
## cost 
cost = cost1 + cost2

In [ ]:
## optimizer
optimizer = tf.train.AdamOptimizer(lr)
gvs = optimizer.compute_gradients(cost)
capped_gvs = [(tf.clip_by_value(grad, -5., 5.), var) for grad, var in gvs]
optimizer = optimizer.apply_gradients(capped_gvs, global_step=global_step)

Evaluacion

In [ ]:
def evaluation(test_R,test_mask_R,Estimated_R,num_test_ratings):

    pre_numerator = np.multiply((test_R - Estimated_R), test_mask_R)
    numerator = np.sum(np.square(pre_numerator))
    denominator = num_test_ratings
    RMSE = np.sqrt(numerator / float(denominator))

    pre_numeartor = np.multiply((test_R - Estimated_R), test_mask_R)
    numerator = np.sum(np.abs(pre_numeartor))
    denominator = num_test_ratings
    MAE = numerator / float(denominator)

    pre_numeartor1 = np.sign(Estimated_R - 0.5)
    tmp_test_R = np.sign(test_R - 0.5)

    pre_numerator2 = np.multiply((pre_numeartor1 == tmp_test_R), test_mask_R)
    numerator = np.sum(pre_numerator2)
    denominator = num_test_ratings
    ACC = numerator / float(denominator)

    a = np.log(Estimated_R)
    b = np.log(1 - Estimated_R)
    a[a == -np.inf] = 0
    b[b == -np.inf] = 0

    tmp_r = test_R
    tmp_r = a * (tmp_r > 0) + b * (tmp_r == 0)
    tmp_r = np.multiply(tmp_r, test_mask_R)
    numerator = np.sum(tmp_r)
    denominator = num_test_ratings
    AVG_loglikelihood = numerator / float(denominator)

    return RMSE,MAE,ACC,AVG_loglikelihood

Entrenar Modelo

In [ ]:
## tf session start
sess = tf.Session()
init = tf.global_variables_initializer()
sess.run(init)

In [ ]:
epoch = 10
train_cost_list = []
test_cost_list = []
test_rmse_list = []
test_mae_list = []
test_acc_list = []
test_avg_loglike_list = []
batch_size = 64
display_step = 1
num_batch = int(num_users / float(batch_size)) + 1

In [ ]:
import time

start_time = time.time()
for itr in range(epoch):
    ## corruption data user shuffle
    corruption_level = 0.3
    mask_corruption_np = np.random.binomial(1, 1 - corruption_level, (num_users, num_items))
    random_perm_doc_idx = np.random.permutation(num_users)
    batch_cost = 0
    for i in range(num_batch):
        if i == num_batch - 1:
            batch_set_idx = random_perm_doc_idx[i * batch_size:]
        elif i < num_batch -1:
            batch_set_idx = random_perm_doc_idx[i * batch_size : (i+1) * batch_size]

        _, Cost = sess.run([optimizer, cost], 
                           feed_dict={model_mask_corruption: mask_corruption_np[batch_set_idx, :],
                                      input_R: train_R[batch_set_idx, :],
                                      input_mask_R: train_mask_R[batch_set_idx, :],
                                      model_batch_data_idx: batch_set_idx})
        batch_cost = batch_cost + Cost

    if i % display_step == 0:
        print ("Training //", "Epoch %d //" % (itr+1),  
               "Train cost = {:.2f}".format(batch_cost/num_batch), 
               "Elapsed time : %d sec" % (time.time() - start_time))
    
    '''test''' 
    ## validation test corruption
    mask_corruption_np = np.random.binomial(1, 1 - 0, (num_users, num_items))
    batch_set_idx = np.arange(num_users)
    
    Cost, decoder = sess.run([cost, Decoder],
                        feed_dict={model_mask_corruption: mask_corruption_np, 
                                   input_R: test_R,
                                   input_mask_R: test_mask_R,
                                   model_batch_data_idx: batch_set_idx})
    test_cost_list.append(Cost)
    Estimated_R = decoder.clip(min=0, max=1)
    RMSE, MAE, ACC, AVG_loglikelihood = evaluation(test_R, test_mask_R, 
                                               Estimated_R, num_test_ratings)
    test_rmse_list.append(RMSE)
    test_mae_list.append(MAE)
    test_acc_list.append(ACC)
    test_avg_loglike_list.append(AVG_loglikelihood)
    
    if itr % display_step == 0:
        print("Testing //", "Epoch %d //" % (itr+1), " Test cost = {:.2f}".format(Cost))
        print("RMSE = {:.4f}".format(RMSE), "MAE = {:.4f}".format(MAE), 
              "ACC = {:.10f}".format(ACC), "AVG Loglike = {:.4f}".format(AVG_loglikelihood))
        print("=" * 100)
        
    if RMSE <= min_RMSE:
        min_RMSE = RMSE
        min_epoch = itr
        patience = 0
    else:
        patience = patience + 1
        
    if (itr > 100) and (patience >= total_patience):
        test_rmse_list.append(test_rmse_list[min_epoch])
        test_mae_list.append(test_mae_list[min_epoch])
        test_acc_list.append(test_acc_list[min_epoch])
        test_avg_loglike_list.append(test_avg_loglike_list[min_epoch])
        earlystop_switch = True
        print ("========== Early Stopping at Epoch %d" %itr+1)